# 06 — Validation and regression workflow


This notebook gives a practical validation workflow for the pure-Python port.

For exact numerical identity with a previous C++ run, the Python package should be tested against **golden reference files** generated by the original workflow. The Python package itself remains pure Python; the golden files are just reference data.

In [ ]:
from pathlib import Path
import sys

# Make the local editable package importable when the notebooks are run from the repo.
HERE = Path.cwd()
CANDIDATES = [HERE / "src", HERE.parent / "src", HERE.parent.parent / "src"]
for p in CANDIDATES:
    if (p / "nucnetpy").exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nucnetpy as nn
print("nucnetpy version:", nn.__version__)
def build_toy_alpha_network():
    """Build a small alpha-chain toy network for tutorials.

    This is not intended to be a physical helium-burning network.  The rates are
    deliberately simple constants so that the examples run quickly and are easy
    to inspect.
    """
    net = nn.Network()
    for name in ["he4", "be8", "c12"]:
        net.add_species(nn.Species.parse(name))

    r1 = nn.Reaction.from_names(
        ["he4", "he4"], ["be8"],
        constant_rate=2.0e-6,
        q_value=0.092,
        label="toy_2alpha_to_be8",
    )
    r2 = nn.Reaction.from_names(
        ["be8", "he4"], ["c12"],
        constant_rate=5.0e-5,
        q_value=7.367,
        label="toy_be8_alpha_to_c12",
    )
    net.reactions.add(r1)
    net.reactions.add(r2)

    zone = nn.Zone(label=("toy", "0", "0"), properties={"t9": "1.0", "rho": "1e4"})
    zone.set_abundance("he4", 0.25)  # X(he4) = A*Y = 1.0
    zone.set_abundance("be8", 0.0)
    zone.set_abundance("c12", 0.0)
    net.add_zone(zone)
    return net
net = build_toy_alpha_network()

## Validate a network

`validate_network` checks missing species, invalid reactions, and basic structural problems.

In [ ]:
report = nn.validate_network(net)
report

## Create a golden result from the current package

During development you can save a known output and compare later versions against it. For C++ identity work, replace this generated table with a table produced from the original C++ code.

In [ ]:
zone = net.zone(0)
times = nn.time_grid(0.0, 2.0e-2, 100)
res = nn.evolve_zone(net, zone, times, nn.constant_thermo(1.0, 1e4), method='bdf', rtol=1e-10)

golden = pd.DataFrame(res.y, columns=res.species)
golden.insert(0, 'time', res.time)
golden_path = Path('data') / 'toy_golden_python.csv'
golden.to_csv(golden_path, index=False)
print(golden_path.resolve())
golden.head()

## Compare a new run to the golden table

Use strict tolerances while developing individual formulas. For large stiff networks, start with a realistic tolerance and tighten it after matching solver conventions, species ordering, constants, and rate interpolation.

In [ ]:
new = nn.evolve_zone(net, zone, times, nn.constant_thermo(1.0, 1e4), method='bdf', rtol=1e-10)
old = pd.read_csv(golden_path)

max_abs = {}
for i, s in enumerate(new.species):
    max_abs[s] = float(np.max(np.abs(new.y[:, i] - old[s].to_numpy())))
max_abs

## Recommended exact-identity checklist

1. Use the same species ordering.
2. Use the same nuclear masses and partition functions.
3. Confirm ReacLib coefficients and rate conventions.
4. Confirm screening model and weak-rate interpolation.
5. Match time grid, tolerances, Jacobian options, and positivity handling.
6. Compare intermediate quantities: rates, flows, `ydot`, Jacobian, then final abundances.